# Incremental Ingestion

Incremental ingestion is the foundation of modern Lakehouse pipelines. This module covers the three primary patterns:

- **COPY INTO** — idempotent, file-level batch loading into Delta tables
- **Auto Loader** (`cloudFiles`) — scalable streaming file ingestion with schema inference
- **Structured Streaming** — trigger modes, output modes, checkpoints and schema rescue

Advanced streaming patterns (stream-static joins, watermarking, `badRecordsPath`, Change Data Feed) are in **[BONUS_05 — Streaming Advanced](BONUS_05_streaming_advanced.ipynb)**.

## Learning Objectives

After completing this module you will be able to:

- **Use** `COPY INTO` to load files idempotently into a Delta table
- **Explain** why COPY INTO is safe to re-run (file tracking) and how to force a reload
- **Configure** Auto Loader (`cloudFiles`) with schema inference and evolution
- **Choose** an Auto Loader file detection mode: directory listing vs file notification (file events)
- **Run** incremental streaming queries with `trigger(availableNow=True)` and explain when `processingTime` triggers apply (classic compute only)
- **Handle** schema changes and malformed data with rescue mode (`_rescued_data`)

| Exam Domain | Weight |
|---|---|
| 2 — Data Ingestion and Loading | **21%** |

## Setup

Initialize the environment, import libraries, and prepare simulated data sources for all ingestion demos in this module.



In [0]:
%run ../../setup/00_setup

### Configuration

Import libraries and define paths for source data, checkpoints, and schema locations.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import time

In [0]:
# [1/2] Define catalog context and all demo paths
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

# Source data (real dataset)
SOURCE_CUSTOMERS = f"{DATASET_PATH}/customers/customers.csv"
SOURCE_ORDERS = f"{DATASET_PATH}/orders/orders_batch.json"

# Demo paths (simulated file arrivals)
DEMO_BASE_PATH = f"{DATASET_PATH}/ingestion_demo"
BATCH_SOURCE_PATH = f"{DEMO_BASE_PATH}/batch_source"
STREAM_SOURCE_PATH = f"{DEMO_BASE_PATH}/stream_source"

# Technical paths
CHECKPOINT_BASE_PATH = f"{DEMO_BASE_PATH}/checkpoints"
SCHEMA_BASE_PATH = f"{DEMO_BASE_PATH}/schemas"

print(f"Paths defined. Base: {DEMO_BASE_PATH}")

In [0]:
# [2/2] Cleanup previous demo run — start fresh
dbutils.fs.rm(DEMO_BASE_PATH, True)
print(f"Demo environment prepared at: {DEMO_BASE_PATH}")

In [0]:
display(
    spark.createDataFrame([
        ("CATALOG", CATALOG),
        ("BRONZE_SCHEMA", BRONZE_SCHEMA),
        ("SILVER_SCHEMA", SILVER_SCHEMA),
        ("USER", raw_user),
        ("CUSTOMERS_CSV", SOURCE_CUSTOMERS),
        ("STREAMING_SOURCE_PATH", STREAM_SOURCE_PATH)
    ], ["Variable", "Value"])
)

In [0]:
# [1/2] Prepare batch data — split customers CSV into 4 daily arrivals
df_customers = spark.read.option("header", "true").csv(SOURCE_CUSTOMERS)
df_batch_day1, df_batch_day2, df_batch_day3, df_batch_day4 = df_customers.randomSplit([0.25]*4, seed=42)

# Save Day 1 immediately — days 2-4 "arrive" later during the demo
df_batch_day1.write.mode("overwrite").option("header", "true").csv(f"{BATCH_SOURCE_PATH}/day1")
print(f"Batch Data: Day 1 ready at {BATCH_SOURCE_PATH}/day1")
print(f"Day 2-4 in memory — will be saved on demand to simulate daily arrivals")

In [0]:
# [2/2] Prepare streaming data — split orders JSON into 20 micro-batches
SOURCE_STREAM_FILES = f"{DATASET_PATH}/orders/stream/*.json"
df_all_orders = spark.read.json(SOURCE_STREAM_FILES)

# Split into 20 parts (5% each) — simulate continuous data arrival
stream_batches = df_all_orders.randomSplit([0.05] * 20, seed=42)

# Save Batch 1 immediately to start the stream
stream_batches[0].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_01")
print(f"Stream Data: Batch 1 ready at {STREAM_SOURCE_PATH}/batch_01")

### Data Loading Methods Overview

| Feature | CTAS | COPY INTO | Auto Loader |
|---------|------|-----------|-------------|
| **Incremental** | No | Yes | Yes |
| **Idempotent** | No | Yes | Yes |
| **Schema Evolution** | No | Limited | Advanced |
| **File Tracking** | No | Metadata | Checkpoint |
| **Scalability** | Low | Medium | High |
| **Streaming** | No | No | Yes |
| **Use Case** | One-time | Scheduled batch | Real-time/Streaming |

> **Exam Tip:** Auto Loader (`cloudFiles`) is the **recommended** ingestion method for new projects. COPY INTO is suitable for scheduled batch loads of up to thousands of files.



## COPY INTO — Batch Loading

> **Databricks official definition:** "COPY INTO is a SQL command that loads data from a file location into a Delta table. COPY INTO is idempotent — files that have already been loaded are skipped on subsequent runs. It is best suited for batch workloads loading up to thousands of files. For millions of files or continuous streaming, use Auto Loader."
> — *[Databricks Documentation](https://docs.databricks.com/en/sql/language-manual/delta-copy-into.html)*

COPY INTO provides idempotent, file-level batch ingestion from cloud storage into Delta tables, automatically tracking which files have already been loaded.

**Syntax:**
```sql
COPY INTO target_table
FROM (SELECT ... FROM 'path/')
FILEFORMAT = { CSV | JSON | PARQUET | AVRO | ORC | TEXT | BINARYFILE }
FORMAT_OPTIONS ('key' = 'value', ...)
COPY_OPTIONS ('key' = 'value', ...)
```

| Clause | Key Options | Description |
|---|---|---|
| `FILEFORMAT` | `CSV`, `JSON`, `PARQUET`, `AVRO` | Source file format |
| `FORMAT_OPTIONS` | `header`, `inferSchema`, `delimiter` | Format-specific parsing options |
| `COPY_OPTIONS` | `mergeSchema`, `force` | `mergeSchema=true` enables schema evolution; `force=true` reloads already-processed files |

> **Pro Tip:** `COPY INTO` is **idempotent** — it tracks processed files and won't load duplicates on re-run.

**COPY INTO — Syntax Reference**

| Element | Syntax |
|---------|--------|
| Basic load | `COPY INTO table FROM '/path/' FILEFORMAT = CSV` |
| With transform | `COPY INTO table FROM (SELECT col, CAST(ts AS TIMESTAMP) FROM '/path/') FILEFORMAT = JSON` |
| Format options | `FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')` |
| Copy options | `COPY_OPTIONS ('mergeSchema' = 'true')` |
| Force re-load | `COPY INTO table FROM '/path/' FILEFORMAT = CSV COPY_OPTIONS ('force' = 'true')` — disables idempotency, reloads every file |
| Supported formats | `CSV` \| `JSON` \| `AVRO` \| `ORC` \| `PARQUET` \| `TEXT` \| `BINARYFILE` |

### Demo: COPY INTO from CSV

We load a CSV file with customer data into a Delta table using `COPY INTO`. We define the target table, add transformations in the `SELECT` (casting, computed columns) and run an idempotent load.

In [0]:
TABLE_CUSTOMERS = f"{BRONZE_SCHEMA}.customers_batch"

**Creating target table:**

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE_CUSTOMERS}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_CUSTOMERS} (
  customer_id STRING,
  first_name STRING,
  last_name STRING,
  email STRING,
  phone STRING,
  city STRING,
  state STRING,
  country STRING,
  registration_date DATE,
  customer_segment STRING,
  _ingestion_timestamp TIMESTAMP
) USING DELTA
COMMENT 'Customers data - Bronze layer'
""")

**Execute COPY INTO:**

> **Note:** the first displayed rows may show `customer_id = null` — the source dataset deliberately contains ~3% rows with a NULL id (data-quality exercises later). The load is not broken.

In [0]:
# Load Day 1 data
result = spark.sql(f"""
COPY INTO {TABLE_CUSTOMERS}
FROM (
  SELECT 
    customer_id,
    first_name,
    last_name,
    email,
    phone,
    city,
    state,
    country,
    TO_DATE(registration_date, 'yyyy-MM-dd') as registration_date,
    customer_segment,
    current_timestamp() as _ingestion_timestamp
  FROM '{BATCH_SOURCE_PATH}/day1'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

display(result)
display(spark.table(TABLE_CUSTOMERS))

### Demo: Idempotency

We run `COPY INTO` again on the same data and check that no duplicates were added — file tracking automatically skips files that have already been loaded.

In [0]:
count_before = spark.table(TABLE_CUSTOMERS).count()

# Re-run COPY INTO (same source path) — expect num_affected_rows = 0
display(spark.sql(f"""
COPY INTO {TABLE_CUSTOMERS}
FROM (
  SELECT 
    customer_id, first_name, last_name, email, phone,
    city, state, country,
    TO_DATE(registration_date, 'yyyy-MM-dd') as registration_date,
    customer_segment,
    current_timestamp() as _ingestion_timestamp
  FROM '{BATCH_SOURCE_PATH}/*'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
"""))

In [0]:
count_after = spark.table(TABLE_CUSTOMERS).count()
display(count_after)

In [0]:
display(
    spark.createDataFrame([
        ("Before", count_before),
        ("After", count_after),
        ("Difference", count_after - count_before)
    ], ["State", "Count"])
)

### Demo: Adding More Days

We simulate data delivered on the following days (day2, day3, day4). After the new files land, `COPY INTO` loads only the new files — day1 is not processed again.

In [0]:
# Add Day 2 data
count_before_days = spark.table(TABLE_CUSTOMERS).count()
print(f"Current row count: {count_before_days}")

df_batch_day2.write.mode("overwrite").option("header", "true").csv(f"{BATCH_SOURCE_PATH}/day2")
print(f"Day 2 data written to {BATCH_SOURCE_PATH}/day2")

In [0]:
# Add Day 3 data
df_batch_day3.write.mode("overwrite").option("header", "true").csv(f"{BATCH_SOURCE_PATH}/day3")
print(f"Day 3 data written to {BATCH_SOURCE_PATH}/day3")

In [0]:
# Add Day 4 data
df_batch_day4.write.mode("overwrite").option("header", "true").csv(f"{BATCH_SOURCE_PATH}/day4")
print(f"Day 4 data written to {BATCH_SOURCE_PATH}/day4")

In [0]:
# Now COPY INTO picks up only the NEW files (day2, day3, day4)
result = spark.sql(f"""
COPY INTO {TABLE_CUSTOMERS}
FROM (
  SELECT 
    customer_id,
    first_name,
    last_name,
    email,
    phone,
    city,
    state,
    country,
    TO_DATE(registration_date, 'yyyy-MM-dd') as registration_date,
    customer_segment,
    current_timestamp() as _ingestion_timestamp
  FROM '{BATCH_SOURCE_PATH}/*'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
""")

count_after_days = spark.table(TABLE_CUSTOMERS).count()
print(f"Count before: {count_before_days}")
print(f"Count after:  {count_after_days}")
print(f"New rows:     {count_after_days - count_before_days}")

display(result)

## Auto Loader — Streaming Ingestion

> **Databricks official definition:** "Auto Loader incrementally and efficiently processes new data files as they arrive in cloud storage. Given an input directory path, the `cloudFiles` source automatically processes new files and provides exactly-once guarantees via checkpointing. It supports schema inference, schema evolution, and scales to billions of files."
> — *[Databricks Documentation](https://docs.databricks.com/en/ingestion/cloud-object-storage/auto-loader/index.html)*

Auto Loader (`cloudFiles`) provides scalable, checkpoint-based streaming ingestion from cloud storage with built-in schema evolution and exactly-once guarantees.

**readStream pattern:**
```python
spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json|csv|parquet|...")
    .option("cloudFiles.schemaLocation", "/path/to/schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns|rescue|failOnNewColumns|none")
    .load("/source/path")
```

| Option | Values | Description |
|---|---|---|
| `cloudFiles.format` | `json`, `csv`, `parquet`, `avro` | Source file format |
| `cloudFiles.schemaLocation` | path | Where Auto Loader persists the inferred schema |
| `cloudFiles.schemaEvolutionMode` | `addNewColumns` (default when **no** schema is provided), `rescue`, `failOnNewColumns`, `none` (default when a schema is provided via `.schema(...)`) | How to handle schema changes |
| `cloudFiles.inferColumnTypes` | `true` / `false` | Infer precise column types (default: all strings) |
| `cloudFiles.includeExistingFiles` | `true` / `false` | Process files that existed before the stream started |
| `cloudFiles.maxFilesPerTrigger` | integer | Limits files processed per micro-batch |

> **Exam Tip:** Auto Loader uses `cloudFiles` format with **checkpoint-based** exactly-once processing. It's the **recommended** method for new ingestion pipelines.



### Trigger Modes & Output Modes

| Trigger Mode | Behavior | Use Case |
|------|----------|----------|
| `availableNow=True` | Process all available → stop | Scheduled jobs (works on serverless and classic) |
| `processingTime="10 seconds"` | Micro-batch every N seconds, runs until stopped | Low-latency continuous jobs — **classic compute only** (not supported on serverless) |
| `once=True` | Legacy (deprecated) | — |

<img src="../../../assets/images/113d4c6273584dc6aa6882e2afe85d0b.png" width="800">

| Output Mode | Description | Use Case |
|------|-------------|----------|
| **Append** | Only new rows | Raw data ingestion (stateless) |
| **Update** | Only updated rows | Aggregations (stateful) |
| **Complete** | Entire result rewritten | Small aggregations |

In [0]:
try:
    dbutils.fs.rm(CHECKPOINT_BASE_PATH, True)
    dbutils.fs.rm(SCHEMA_BASE_PATH, True)
except:
    pass

In [0]:
TARGET_TABLE_AL = f"{BRONZE_SCHEMA}.orders_autoloader"
CHECKPOINT_AL = f"{CHECKPOINT_BASE_PATH}/autoloader"
SCHEMA_AL = f"{SCHEMA_BASE_PATH}/autoloader"

**Auto Loader readStream configuration:**

### Auto Loader Configuration Options

| Category | Key Options |
|---|---|
| **Common** | `cloudFiles.format`, `cloudFiles.schemaLocation`, `cloudFiles.includeExistingFiles` |
| **Schema** | `cloudFiles.inferColumnTypes`, `cloudFiles.schemaEvolutionMode` (`addNewColumns`, `rescue`, `failOnNewColumns`, `none`) |
| **Rate limiting** | `cloudFiles.maxFilesPerTrigger`, `cloudFiles.maxBytesPerTrigger` |
| **File detection** | `cloudFiles.useManagedFileEvents` (file events, recommended), `cloudFiles.useNotifications` (legacy notifications: Event Grid/Queue Storage, SNS/SQS, Pub/Sub) |

[Full options reference](https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/auto-loader/options/)

### File Detection Modes — Directory Listing vs File Notification

| Mode | How new files are discovered | Enable with | Notes |
|---|---|---|---|
| **Directory listing** (default) | Lists the input directory on every micro-batch | nothing — default | Minimal permissions, quick to start; listing cost grows with the number of files |
| **File notification with file events** (recommended) | Reads the file-event cache Unity Catalog keeps for an **external location** that has file events enabled | `.option("cloudFiles.useManagedFileEvents", "true")` | No queues or extra cloud permissions in the stream; DBR 14.3 LTS+; scales to millions of files |
| **Legacy file notification** | Subscribes to cloud storage events through a queue (Azure Event Grid + Queue Storage, AWS SNS/SQS, GCP Pub/Sub) | `.option("cloudFiles.useNotifications", "true")` | Needs permissions to create/read the queue; still supported, new workloads should use file events |

> **Exam Tip:** Auto Loader's file discovery modes are **directory listing** (default, simplest) and **file notification** (event-driven, for very large or high-volume directories). `cloudFiles.useIncrementalListing` is deprecated — Databricks recommends file notification with file events instead. This demo uses directory listing on a Volume path.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE_AL}")

In [0]:
df_autoloader = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_AL)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(STREAM_SOURCE_PATH) # Reading from our simulated stream source
)

**Adding metadata columns:**

In [0]:
from pyspark.sql.functions import col

df_enriched = (df_autoloader
    .withColumn("_processing_time", F.current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

**Start streaming with `availableNow` trigger:**

> `availableNow` - processes all available data and stops (batch-like streaming)
>
> **Note:** rows with `customer_id = null` near the top are expected — ~3% of the orders have a NULL customer id in the source data.

In [0]:
query = (df_enriched.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_AL)
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE_AL)
)
query.awaitTermination()   # availableNow stops by itself — wait before reading the table
display(spark.table(f"{TARGET_TABLE_AL}"))

**Auto Loader results:**

In [0]:
display(
    spark.createDataFrame([
        ("Records Loaded", str(spark.table(TARGET_TABLE_AL).count())),
        ("Source Files", str(spark.table(TARGET_TABLE_AL).select("_source_file").distinct().count()))
    ], ["Metric", "Value"])
)

### Demo: Incremental Processing — Add New Data

A new batch lands in the source directory. Re-running the **same** stream with the **same** checkpoint picks up only the new files; a run with nothing new adds 0 rows.

In [0]:
# Batch 2 arrives in the landing directory
stream_batches[1].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_02")
print(f"Batch 2 written to {STREAM_SOURCE_PATH}/batch_02")

In [0]:
def run_autoloader_once(label):
    """One scheduled incremental run: same stream definition + same checkpoint, availableNow trigger."""
    before = spark.table(TARGET_TABLE_AL).count()
    q = (df_enriched.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_AL)
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE_AL)
    )
    q.awaitTermination()
    after = spark.table(TARGET_TABLE_AL).count()
    print(f"{label:<32} +{after - before:>6,} new rows   (total {after:,})")

run_autoloader_once("Run after batch_02 arrived")
run_autoloader_once("Run again, nothing new")

### Demo: Repeated Scheduled Runs (`availableNow`)

In production an ingestion job usually runs on a schedule (e.g., every 15 minutes) with `trigger(availableNow=True)`: each run processes whatever arrived since the last checkpoint and then stops, so compute is not kept running between runs. Below, three more batches arrive one by one and a run follows each arrival.

> **Classic compute only:** a continuously running stream uses a time-based trigger, e.g. `.trigger(processingTime="10 seconds")`, and runs until `query.stop()`. Serverless compute supports only `availableNow` (and the deprecated `once`) — a `processingTime` trigger raises an error there.

In [0]:
# Simulate arrival of batches 3-5, with one scheduled run after each arrival
for i in range(2, 5):
    batch_num = i + 1
    stream_batches[i].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_{batch_num:02d}")
    run_autoloader_once(f"Run after batch_{batch_num:02d} arrived")

In [0]:
# Rows per source batch directory — every batch loaded exactly once
display(
    spark.table(TARGET_TABLE_AL)
    .withColumn("batch_dir", F.regexp_extract("_source_file", r"(batch_\d+)", 1))
    .groupBy("batch_dir").count()
    .orderBy("batch_dir")
)

## Error Handling

Databricks provides multiple strategies for handling malformed, corrupted, or schema-mismatched records during ingestion.

| Mode | Behavior |
|------|----------|
| `PERMISSIVE` | Parses what it can, errors → `_corrupt_record` |
| `DROPMALFORMED` | Removes malformed records |
| `FAILFAST` | Stops on first error |

> **Best Practice:** Use `badRecordsPath` to save malformed records for later analysis.
>
> This section is reference only — the runnable `badRecordsPath` demo is in **[BONUS_05 — Streaming Advanced](BONUS_05_streaming_advanced.ipynb)**. Below we demo the Auto Loader alternative: rescue mode.



### Schema Evolution & Rescued Data

| `schemaEvolutionMode` | Behavior |
|---|---|
| `addNewColumns` | Automatically adds new columns — **default when no schema is provided** |
| `rescue` | New/mismatched → `_rescued_data` JSON column |
| `failOnNewColumns` | Fail if schema changes |
| `none` | Ignores new columns — **default when a schema is provided** via `.schema(...)` |

> **Exam Tip:** `_rescued_data` column captures new columns, type mismatches, and malformed records when using `rescue` mode. With an explicit `.schema(...)` the default mode is `none`, so `rescue` must be set explicitly (as below).



In [0]:
TARGET_TABLE_RESCUE = f"{BRONZE_SCHEMA}.orders_rescued"
CHECKPOINT_RESCUE = f"{CHECKPOINT_BASE_PATH}/rescue"
SCHEMA_RESCUE = f"{SCHEMA_BASE_PATH}/rescue"

spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE_RESCUE}")

**Define explicit schema (partial):**

In [0]:
# Deliberately define only some columns - rest will go to _rescued_data
partial_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("total_amount", DoubleType(), True)
])

**Auto Loader with rescue mode:**

In [0]:
# Create BAD data (Extra column + Type mismatch)
bad_data = [{"order_id": 99999, "total_amount": "INVALID_NUMBER", "new_col": "surprise"}]
spark.createDataFrame(bad_data).write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/bad_data")

In [0]:
df_rescue = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_RESCUE)
    .option("cloudFiles.schemaEvolutionMode", "rescue")  # Rescue mode!
    .schema(partial_schema)  # Partial schema
    .load(STREAM_SOURCE_PATH)
)

**Start stream:**

In [0]:
query_rescue = (df_rescue.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_RESCUE)
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE_RESCUE)
)
query_rescue.awaitTermination()
display(spark.table(TARGET_TABLE_RESCUE))

**Schema with `_rescued_data` column:**

In [0]:
spark.table(TARGET_TABLE_RESCUE).printSchema()

**The bad record in `_rescued_data`:**

> With the partial schema *every* row has `_rescued_data` (columns like `product_id` are not in the schema). Filter on `new_col` to see the injected record: the unexpected column and the `"INVALID_NUMBER"` value (type mismatch → `total_amount` is `null`) are preserved as JSON.

In [0]:
display(
    spark.table(TARGET_TABLE_RESCUE)
    .filter(F.col("_rescued_data").contains("new_col"))   # the injected bad record
)

## Lakeflow Connect (Informational)

> ⏭️ **Self-study** — skipped in class; not required by any lab. Lakeflow Connect is demonstrated live in `02a_lakeflow_connect_demo`.

Zero-code SaaS ingestion via UI: Salesforce, Workday, HubSpot, SAP, ServiceNow, etc.

| Method | Use Case |
|--------|----------|
| **COPY INTO** | Files in cloud storage (batch) |
| **Auto Loader** | Files in cloud storage (streaming) |
| **Lakeflow Connect** | Data from SaaS systems |
| **Lakeflow Spark Declarative Pipelines** | Transformations Bronze → Silver → Gold |



## Summary

| Topic | Key Concept | Exam Keywords |
|---|---|---|
| **COPY INTO** | Idempotent batch loading, file tracking | `COPY INTO`, `FILEFORMAT`, `COPY_OPTIONS ('mergeSchema' = 'true')`, `'force' = 'true'` |
| **Auto Loader** | `cloudFiles` streaming ingestion | `cloudFiles.format`, `schemaLocation`, `schemaEvolutionMode` |
| **File detection modes** | Directory listing (default) vs file notification | `cloudFiles.useManagedFileEvents`, `cloudFiles.useNotifications` |
| **Trigger Modes** | `availableNow=True` (serverless + classic), `processingTime` (classic only) | Scheduled incremental vs continuous |
| **Schema Evolution** | `rescue`, `addNewColumns`, `failOnNewColumns` | `_rescued_data` column |
| **Lakeflow Connect** | Zero-code SaaS ingestion | Salesforce, Workday, SAP |

> Stream-static joins, watermarking, `badRecordsPath` and CDF for incremental ETL: **[BONUS_05 — Streaming Advanced](BONUS_05_streaming_advanced.ipynb)**

In [0]:
# List of created tables
created_tables = [
    "customers_batch",
    "orders_autoloader",
    "orders_rescued",
]

## Cleanup

Remove demo tables, checkpoints, and temporary data created during this module.



In [0]:
results = []
for table in created_tables:
    full_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table}"
    try:
        if spark.catalog.tableExists(full_table):
            count = spark.table(full_table).count()
            results.append((table, "EXISTS", str(count)))
        else:
            results.append((table, "NOT FOUND", "-"))
    except Exception as e:
        results.append((table, "ERROR", str(e)[:30]))

display(spark.createDataFrame(results, ["Table", "Status", "Records"]))

In [0]:
# Cleanup flag
CLEANUP_ENABLED = False

**Execute cleanup (if enabled):**

In [0]:
if CLEANUP_ENABLED:
    results = []
    for table in created_tables:
        full_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table}"
        try:
            spark.sql(f"DROP TABLE IF EXISTS {full_table}")
            results.append((table, "DROPPED"))
        except Exception as e:
            results.append((table, f"ERROR: {str(e)[:30]}"))
    
    # Cleanup checkpoints
    try:
        dbutils.fs.rm(CHECKPOINT_BASE_PATH, True)
        results.append(("checkpoints", "REMOVED"))
    except:
        results.append(("checkpoints", "NOT FOUND"))
    
    display(spark.createDataFrame(results, ["Resource", "Status"]))
else:
    display(spark.createDataFrame([
        ("CLEANUP_ENABLED", "False"),
        ("Action", "Change to True to delete resources")
    ], ["Setting", "Value"]))

← [04 — Delta Optimization](04_delta_optimization.ipynb) | **[ README](../../../README.md)** | [06 — Medallion Architecture →](06_medallion_architecture.ipynb)